In [1]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import os
from dotenv import load_dotenv
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from typing import Dict, List

c:\Users\cosim\Desktop\Cosima\Studium\DataScience\Semester2\SocialMediaAnalytics\SMAEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Social Media Analytics Capstone

### Cosima Baumeier, 950619

### Data preparation

In [2]:
df = pd.read_csv('goodreads_rag_dataset.csv') #reading in the data created in dataengineering.ipynb

In [6]:
def aggregate(group):
    review_texts = group['review_text'].str.cat(sep="\n---\n") #seperating the reviews with ---
    return pd.Series({
        "book_id": group['book_id'].iloc[0],
        "title": group['title'].iloc[0],
        "author": group['author_name'].iloc[0],
        "full_book_context": review_texts
    })

df_final_docs = df_subset.groupby('book_id').apply(aggregate).reset_index(drop=True)
#using the function -> new dataframe with four columns: book_idm title, author and full_book_context

C:\Users\cosim\AppData\Local\Temp\ipykernel_8296\3763668291.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_final_docs = df_subset.groupby('book_id').apply(aggregate).reset_index(drop=True)


### Splitting into chunks

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, #one chunk has a maximum of 500 characters
    chunk_overlap=50, #chunks have 50 overlapping characters so context is not lost
    length_function=len, #uses the length of the string (number of characters)
    separators=["\n---\n", "\n\n", "\n", " ", ""] ) #see below

The text splitter tries to separate the chunks between each review ('---'). If the chunk is still too big, it separates after a paragraph. If that is still to big, after a line breaks. If it is still too large, then between words. The last resort is between characters.

In [8]:
all_chunks = [] #creating a list for the chunks
for index, row in df_final_docs.iterrows():
    metadata = {
        "book_id": row["book_id"],
        "title": row["title"],
        "author": row["author"]
    }
    chunks = text_splitter.create_documents(
        texts=[row["full_book_context"]],
        metadatas=[metadata] #saving the book_id, title and author with each chunk
    )
    all_chunks.extend(chunks)

### Embedding

In [9]:
load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")

embedding_openai = OpenAIEmbeddings(model="text-embedding-3-small", api_key=api_key, openai_api_base="https://openrouter.ai/api/v1")

### Vector data base

In [10]:
#vectordb = Chroma.from_documents(documents=all_chunks, embedding=embedding_openai, persist_directory="./chroma_db")

In [11]:
vectordb = Chroma(persist_directory="./chroma_db", embedding_function=embedding_openai) #loading the vector data base from the directory

C:\Users\cosim\AppData\Local\Temp\ipykernel_8296\2273804566.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory="./chroma_db", embedding_function=embedding_openai) #loading the vector data base from the directory


### Retriever

In [12]:
retriever = vectordb.as_retriever(search_kwargs={"k": 20}) #the retriever looks for the 20 chunks with the closest vectors

### LLM

In [13]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1")


In [14]:
class RecState(Dict):
    user_query: str
    preferences: str
    retrieved_docs: List
    recommendations: str

In [15]:
def extract_preferences(state: RecState):

    prompt = f"""
    Analyze the user's book preferences based on their message:

    "{state['user_query']}"

    Extract the following:
    - what the user liked
    - what the user disliked
    - desired themes, tropes, tones, vibes, genres
    - specific elements they want more or less of

    Output a concise preference profile.
    """

    response = llm.invoke(prompt)
    state["preferences"] = response
    return state

In [16]:
def retrieve_books(state: RecState):

    query = f"""
    Find books that match these user preferences:

    {state['preferences']}
    """

    docs = retriever.invoke(query)
    
    seen = set()
    unique_docs = []
    for doc in docs: #building a loop that checks if the same book_id appears in multiple chunks -> if yes, it only keeps the first (most relevant) chunk
        book_id = doc.metadata["book_id"]
        if book_id not in seen:
            unique_docs.append(doc)
            seen.add(book_id)

    state["retrieved_docs"] = unique_docs
    return state

In [20]:
def generate_recommendations(state: RecState):

    user_query = state["user_query"]
    context = "\n\n".join([d.page_content for d in state["retrieved_docs"]])

    prompt = f"""
    USER QUERY:
    {user_query}

    PREFERENCE ANALYSIS:
    {state['preferences']}

    RELEVANT BOOK DATA:
    {context}

    Based on this information, recommend 3-5 books.
    For each recommendation:
    - explain why it matches the user’s preferences ("Why it matches:")
    - highlight specific themes, elements, or vibes ("Themes and elements:")
    Keep the tone friendly and helpful.
    """

    response = llm.invoke(prompt)
    state["recommendations"] = response
    return state


In [21]:
rag_graph = (
    StateGraph(RecState)
    .add_node("extract", extract_preferences)
    .add_node("retrieve", retrieve_books)
    .add_node("recommend", generate_recommendations)
    .set_entry_point("extract")
    .add_edge("extract", "retrieve")
    .add_edge("retrieve", "recommend")
    .add_edge("recommend", END)
    .compile()
)


In [22]:
result = rag_graph.invoke({
    "user_query": "I like the setting and characters of Game of Thrones but I don't like how brutal it is"
})

print(result["recommendations"].content)



Based on your preferences for fantasy that emphasizes setting and character development without the brutality of *Game of Thrones*, here are some book recommendations that might suit your taste:

### 1. **The Lies of Locke Lamora** by Scott Lynch
- **Why it matches:** This book features a richly built world and a cast of complex, charismatic characters, much like *Game of Thrones* but with a more light-hearted and whimsical approach. The brutality and graphic violence are significantly toned down.
- **Themes and elements:** The story revolves around a group of con artists in a fantastical version of Venice. You'll find themes of friendship, clever schemes, and a detailed heist plot that provides plenty of adventure without excessive violence.

### 2. **The Goblin Emperor** by Katherine Addison
- **Why it matches:** This standalone novel focuses heavily on character development, political intrigue, and a deeply immersive world. It avoids typical fantasy tropes, particularly violence, ma